# Session 13 — Pose Estimation

**Computer Vision (CVI4IC) · Summer Semester 2026 · FH Upper Austria**

This notebook is **practical-heavy** — most of the lecture's ideas come alive here:

1. **Two modern off-the-shelf models** side by side: YOLO11-pose (single-stage) + ViTPose (top-down, our deep dive).
2. **Decode raw heatmaps** yourself to see how keypoints come out of a network.
3. **Joint angles** + a tiny standing/sitting classifier.
4. **🕺 Mini project — Pose Karaoke:** upload any short video, watch the skeleton dance, get a *dance signature*.
5. **Exercises** at the end.

> **Colab tip:** *Runtime → Change runtime type → T4 GPU*. CPU works too — ViTPose just becomes slow.


## 0 · Setup

Three libraries: `mediapipe`, `ultralytics`, `transformers` (for ViTPose). Plus the usual.

In [ ]:
!pip install -q ultralytics transformers
# torch + torchvision are preinstalled on Colab.

In [ ]:
import os, math, json, time, urllib.request
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "·", "GPU" if DEVICE == "cuda" else "CPU only")
print("OpenCV:", cv2.__version__)

### Grab a few demo images

Four CC-licensed photos hosted on **[Digital-Media/cv_data](https://github.com/Digital-Media/cv_data/tree/main/pose_estimation)** — a ski jumper, a yoga pose, a runner, and a cycling peloton (multi-person).

In [ ]:
CV_DATA = "https://raw.githubusercontent.com/Digital-Media/cv_data/main/pose_estimation"
DEMO_URLS = {
    "skier.jpg":    f"{CV_DATA}/skier.jpg",      # ski jumper mid-air — single-person action
    "yoga.jpg":     f"{CV_DATA}/yoga.jpg",       # silhouette warrior pose — single person
    "runner.jpg":   f"{CV_DATA}/runner.jpg",     # runner on trail — single person (rear view)
    "cyclists.jpg": f"{CV_DATA}/cyclists.jpg",   # cycling peloton — multi-person scene
}
IMGS = Path("imgs"); IMGS.mkdir(exist_ok=True)
for name, url in DEMO_URLS.items():
    p = IMGS / name
    if not p.exists():
        urllib.request.urlretrieve(url, p)
        print(f"  fetched {name}")
print("Files:", sorted(p.name for p in IMGS.glob("*")))


In [ ]:
# Show the demo images
files = sorted(IMGS.glob("*.jpg"))
fig, axes = plt.subplots(1, len(files), figsize=(4 * len(files), 4))
if len(files) == 1: axes = [axes]
for ax, f in zip(axes, files):
    im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    ax.imshow(im); ax.axis("off"); ax.set_title(f.name)
plt.tight_layout(); plt.show()

---
## 1 · YOLO11-pose — single-stage, deployable

One network, one forward pass: persons + keypoints together. **17 COCO keypoints** per detected person.

In [ ]:
from ultralytics import YOLO

yolo_pose = YOLO("yolo11n-pose.pt")     # nano — fast; bump to s/m/l/x for accuracy

def run_yolo_pose(img_bgr):
    res = yolo_pose(img_bgr, imgsz=640, conf=0.3, verbose=False)[0]
    return res                              # has .plot(), .keypoints, .boxes

img = cv2.imread(str(IMGS / "skier.jpg"))
res = run_yolo_pose(img)
ann = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
print(f"YOLO11-pose found {len(res.boxes)} person(s)")
plt.figure(figsize=(6, 7))
plt.imshow(ann); plt.title("YOLO11n-pose"); plt.axis("off")
plt.tight_layout(); plt.show()

---
## 2 · ViTPose — our deep dive

The plain-ViT model from the lecture. HuggingFace ships a clean `ViTPoseForPoseEstimation` wrapper, plus a `PersonImage` detector for stage 1.

> ViTPose is **top-down two-stage** — we need person boxes first. We use HuggingFace's recommended **RT-DETRv2** detector.

In [ ]:
from transformers import (
    AutoProcessor, RTDetrForObjectDetection,
    VitPoseImageProcessor, VitPoseForPoseEstimation,
)
from PIL import Image

# ---- Stage 1: person detector ----
det_id  = "PekingU/rtdetr_v2_r18vd"
det_proc = AutoProcessor.from_pretrained(det_id)
det_model = RTDetrForObjectDetection.from_pretrained(det_id).to(DEVICE).eval()

# ---- Stage 2: keypoint network ----
pose_id = "usyd-community/vitpose-base-simple"
pose_proc  = VitPoseImageProcessor.from_pretrained(pose_id)
pose_model = VitPoseForPoseEstimation.from_pretrained(pose_id).to(DEVICE).eval()
print("ViTPose stack ready ·", pose_id)

In [ ]:
def detect_persons(pil_img, conf=0.5):
    """Run RT-DETR, return boxes (xyxy pixel) of class=person only."""
    inputs = det_proc(images=pil_img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = det_model(**inputs)
    target_size = torch.tensor([pil_img.size[::-1]]).to(DEVICE)
    results = det_proc.post_process_object_detection(outputs, threshold=conf,
                                                       target_sizes=target_size)[0]
    person_boxes = []
    for box, label in zip(results["boxes"], results["labels"]):
        # COCO class id 0 = person
        if det_model.config.id2label[label.item()] == "person":
            person_boxes.append(box.cpu().numpy())
    return np.array(person_boxes) if person_boxes else np.zeros((0, 4))


def run_vitpose(pil_img):
    """Returns: list of (boxes_xyxy, keypoints_array (17, 3)) per detected person."""
    boxes = detect_persons(pil_img)
    if len(boxes) == 0:
        return []
    # ViTPose expects boxes in xywh
    boxes_xywh = np.stack([boxes[:, 0], boxes[:, 1],
                            boxes[:, 2] - boxes[:, 0],
                            boxes[:, 3] - boxes[:, 1]], axis=1)
    inputs = pose_proc(pil_img, boxes=[boxes_xywh], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = pose_model(**inputs)
    pose_results = pose_proc.post_process_pose_estimation(outputs, boxes=[boxes_xywh])
    return [(box, p) for box, p in zip(boxes, pose_results[0])]


pil = Image.open(IMGS / "skier.jpg").convert("RGB")
results = run_vitpose(pil)
print(f"ViTPose found {len(results)} person(s)")

In [ ]:
# Visualize ViTPose result
fig, ax = plt.subplots(figsize=(6, 7))
ax.imshow(pil)
COCO_SKEL = [(15,13),(13,11),(16,14),(14,12),(11,12),(5,11),(6,12),(5,6),
             (5,7),(7,9),(6,8),(8,10),(1,2),(0,1),(0,2),(1,3),(2,4),(0,5),(0,6)]
for box, person in results:
    x1, y1, x2, y2 = box
    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="yellow", lw=2))
    kp = person["keypoints"].cpu().numpy()           # (17, 2)
    sc = person["scores"].cpu().numpy()              # (17,)
    for a, b in COCO_SKEL:
        if sc[a] > 0.3 and sc[b] > 0.3:
            ax.plot([kp[a, 0], kp[b, 0]], [kp[a, 1], kp[b, 1]], "-", color="cyan", lw=2.5)
    for i, (x, y) in enumerate(kp):
        if sc[i] > 0.3:
            ax.plot(x, y, "o", markersize=7, mfc="orange", mec="black", mew=1.0)
ax.axis("off"); ax.set_title("ViTPose-base"); plt.tight_layout(); plt.show()

---
## 3 · Build it yourself — decode raw heatmaps

The lecture talked about heatmaps. Here we inspect them on a ViTPose output. Hook the model to grab the raw output, then do argmax + sub-pixel refinement by hand.

In [ ]:
# Re-run ViTPose but keep the raw heatmap output
pil = Image.open(IMGS / "skier.jpg").convert("RGB")
boxes = detect_persons(pil)
if len(boxes) == 0:
    print("No person detected — try a different image"); raise SystemExit

# Take the first person and run only the keypoint head
boxes_xywh = np.array([[boxes[0, 0], boxes[0, 1],
                        boxes[0, 2] - boxes[0, 0],
                        boxes[0, 3] - boxes[0, 1]]])
inputs = pose_proc(pil, boxes=[boxes_xywh], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = pose_model(**inputs, output_hidden_states=False)

heatmaps = outputs.heatmaps[0].cpu().numpy()    # (17, H, W) — typically 64×48
print("Heatmap stack shape:", heatmaps.shape)

# Show the first 6 heatmaps side by side
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
NAMES_17 = ["nose","l-eye","r-eye","l-ear","r-ear","l-sh","r-sh","l-el","r-el",
             "l-wr","r-wr","l-hip","r-hip","l-kn","r-kn","l-an","r-an"]
for ax, k in zip(axes.ravel(), [0, 5, 7, 9, 11, 13]):
    ax.imshow(heatmaps[k], cmap="hot"); ax.set_title(f"k={k}  {NAMES_17[k]}"); ax.axis("off")
    y, x = np.unravel_index(heatmaps[k].argmax(), heatmaps[k].shape)
    ax.plot(x, y, "c+", markersize=15, mew=2)
plt.suptitle("ViTPose raw heatmaps  ·  cyan + = argmax peak"); plt.tight_layout(); plt.show()

In [ ]:
# Decode by hand: argmax + sub-pixel refinement
def decode_heatmaps(hm, box_xywh):
    """hm: (K, H, W) numpy array.  box_xywh: (4,) pixel coords of crop box.
       Returns (K, 2) keypoints in original-image pixel coordinates."""
    K, H, W = hm.shape
    bx, by, bw, bh = box_xywh
    keypoints = np.zeros((K, 2))
    for i in range(K):
        # Argmax
        y, x = np.unravel_index(hm[i].argmax(), hm[i].shape)
        # Sub-pixel offset (Newell et al.): quarter-pixel from the sign of the derivative
        dx = 0.25 * np.sign(hm[i, y, min(x+1, W-1)] - hm[i, y, max(x-1, 0)])
        dy = 0.25 * np.sign(hm[i, min(y+1, H-1), x] - hm[i, max(y-1, 0), x])
        # Heatmap coords → crop pixels → original image pixels
        keypoints[i] = [bx + (x + dx) * bw / W,
                        by + (y + dy) * bh / H]
    return keypoints

manual_kp = decode_heatmaps(heatmaps, boxes_xywh[0])
print("First 3 decoded keypoints (x, y):")
print(manual_kp[:3])

# Compare to the model's own decoder output
arr = np.array(pil)
fig, ax = plt.subplots(figsize=(6, 7))
ax.imshow(arr)
for x, y in manual_kp:
    ax.plot(x, y, "o", mfc="cyan", mec="black", mew=1, markersize=10)
ax.set_title("Hand-decoded keypoints (argmax + ¼-pixel refinement)"); ax.axis("off")
plt.tight_layout(); plt.show()

---
## 4 · Joint angles — from keypoints to semantics

The dot-product gives angles between bones:

$$\theta = \arccos\!\left(\frac{\mathbf{v}_1 \cdot \mathbf{v}_2}{\|\mathbf{v}_1\|\,\|\mathbf{v}_2\|}\right)$$

Common ones in fitness / physio:
- **Elbow angle** — shoulder–elbow–wrist
- **Knee angle** — hip–knee–ankle
- **Hip angle** — shoulder–hip–knee

In [ ]:
def joint_angle(p1, p2, p3):
    """Angle at p2 (in degrees) formed by p1–p2–p3."""
    v1 = np.array(p1) - np.array(p2)
    v2 = np.array(p3) - np.array(p2)
    cos = np.clip(v1.dot(v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8), -1, 1)
    return float(np.degrees(np.arccos(cos)))

# Get the first person's keypoints
kp = results[0][1]["keypoints"].cpu().numpy()
angles = {
    "left elbow":   joint_angle(kp[5],  kp[7],  kp[9]),
    "right elbow":  joint_angle(kp[6],  kp[8],  kp[10]),
    "left knee":    joint_angle(kp[11], kp[13], kp[15]),
    "right knee":   joint_angle(kp[12], kp[14], kp[16]),
    "left hip":     joint_angle(kp[5],  kp[11], kp[13]),
    "right hip":    joint_angle(kp[6],  kp[12], kp[14]),
}
for name, a in angles.items():
    print(f"  {name:14s} {a:6.1f}°")

### A tiny posture classifier

Knee fully extended (>160°) → standing. Knee bent (<120°) → sitting / squatting. Try it on a few angles:

In [ ]:
def posture(kp):
    """Classify standing / sitting / something-else from leg + hip angles."""
    lk = joint_angle(kp[11], kp[13], kp[15])
    rk = joint_angle(kp[12], kp[14], kp[16])
    avg_knee = (lk + rk) / 2
    if avg_knee > 160: return f"standing  (knee = {avg_knee:.0f}°)"
    if avg_knee < 110: return f"sitting / squatting  (knee = {avg_knee:.0f}°)"
    return f"transition  (knee = {avg_knee:.0f}°)"

print(posture(kp))

---
## 5 · 🕺 Mini project — Pose Karaoke

The fun one. Upload **any short video** — a TikTok dance, a sports clip, you doing the floss, anything 5–30 seconds — and we:

1. Run **YOLO11-pose** on every frame (fast, deployable, no person-detection needed).
2. Produce two output videos:
   - **Overlay** — the original video with the skeleton drawn on top.
   - **Skeleton only** — black background, just the dancing skeleton.
3. Plot the **dance signature** — knee + elbow joint angles over time, your unique movement fingerprint.

> If you don't want to upload anything, the cell below synthesises a small "robot dance" video so the whole pipeline still runs end-to-end.


In [ ]:
# --- Helpers for video pose tracking ---
import pandas as pd
from IPython.display import display

# COCO skeleton edges + colours (re-used from §3)
COCO_SKEL = [(15,13),(13,11),(16,14),(14,12),(11,12),(5,11),(6,12),(5,6),
             (5,7),(7,9),(6,8),(8,10),(1,2),(0,1),(0,2),(1,3),(2,4),(0,5),(0,6)]

def draw_skeleton_cv2(canvas, kp, sc=None, thresh=0.3):
    """Draw COCO skeleton onto a BGR canvas in-place."""
    for a, b in COCO_SKEL:
        if sc is not None and (sc[a] < thresh or sc[b] < thresh):
            continue
        cv2.line(canvas,
                 (int(kp[a, 0]), int(kp[a, 1])),
                 (int(kp[b, 0]), int(kp[b, 1])),
                 (0, 230, 255), 3)
    for i, (x, y) in enumerate(kp):
        if sc is not None and sc[i] < thresh:
            continue
        cv2.circle(canvas, (int(x), int(y)), 4, (255, 165, 0), -1)
        cv2.circle(canvas, (int(x), int(y)), 4, (255, 255, 255), 1)

def show_video_frames(path, n_frames=6):
    """Sample n frames evenly along the video and show as a grid.
    Reliable in Colab (no codec / data-URL headaches). The full MP4
    is still written to disk — open it via the Files tab on the left."""
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
    cols = min(n_frames, 3); rows = (n_frames + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 2.6))
    axes = np.array(axes).ravel()
    for ax, i in zip(axes, idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if ok:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"frame {i}", fontsize=9)
        ax.axis("off")
    cap.release()
    plt.tight_layout(); plt.show()

def process_pose_video(in_path, out_path, mode="overlay", max_frames=300):
    """Run YOLO11-pose on every frame, write annotated video.

    Returns (n_frames_processed, angles_df).
    mode='overlay'        → skeleton on top of the original frame
    mode='skeleton_only'  → skeleton on a black background
    """
    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {in_path}")
    fps  = cap.get(cv2.CAP_PROP_FPS) or 24
    W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    vw   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    angles = []
    n = 0
    while n < max_frames:
        ok, frame = cap.read()
        if not ok: break
        res = yolo_pose(frame, imgsz=640, conf=0.3, verbose=False)[0]

        if mode == "overlay":
            canvas = frame.copy()
        else:
            canvas = np.zeros_like(frame)

        # Draw the first detected person (focus on solo dancer)
        if res.keypoints is not None and len(res.keypoints):
            kp = res.keypoints.xy[0].cpu().numpy()
            sc = (res.keypoints.conf[0].cpu().numpy()
                  if res.keypoints.conf is not None else np.ones(17))
            draw_skeleton_cv2(canvas, kp, sc)
            # Record joint angles for the signature plot
            try:
                angles.append({
                    "frame":   n,
                    "l_knee":  joint_angle(kp[11], kp[13], kp[15]),
                    "r_knee":  joint_angle(kp[12], kp[14], kp[16]),
                    "l_elbow": joint_angle(kp[5],  kp[7],  kp[9]),
                    "r_elbow": joint_angle(kp[6],  kp[8],  kp[10]),
                })
            except Exception:
                pass

        vw.write(canvas)
        n += 1

    cap.release(); vw.release()
    return n, pd.DataFrame(angles)


In [ ]:
# --- Pick a video to dance with ---
#
# By default we use a public Apache-licensed clip of a person walking in front of
# the camera (full body, real human, all 17 keypoints visible).
#
# 👉  STUDENTS: replace this with YOUR OWN clip for the real karaoke experience!
#     Two ways:
#       (a) Run this cell, then click "Choose Files" and pick any short MP4
#           from your phone / laptop (5–30 s works best).
#       (b) Edit DEFAULT_URL below to point at any direct MP4 link.

DEFAULT_URL = (
    "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/"
    "master/face-demographics-walking-and-pause.mp4"
)
SAMPLE = "karaoke_input.mp4"
in_video = None

# (1) Try Colab's upload widget — silently skipped if not on Colab.
try:
    from google.colab import files
    print("👉  Upload your own short MP4 (recommended) — or press Cancel to use the sample.")
    uploaded = files.upload()
    if uploaded:
        in_video = list(uploaded.keys())[0]
        print(f"Using your upload: {in_video}")
except Exception:
    pass

# (2) Otherwise grab the default sample.
if in_video is None:
    print(f"Downloading sample clip → {SAMPLE} …")
    urllib.request.urlretrieve(DEFAULT_URL, SAMPLE)
    in_video = SAMPLE
    print(f"  ok ({Path(SAMPLE).stat().st_size/1024/1024:.1f} MB)")

print("Input video:", in_video)


In [ ]:
# --- Run pose tracking on every frame ---
print("Processing overlay video (this can take ~20–60 s) …")
n_over, signature = process_pose_video(in_video, "with_skeleton.mp4", mode="overlay")
print(f"  overlay:        wrote {n_over} frames  →  with_skeleton.mp4")

print("Processing skeleton-only video …")
n_skel, _ = process_pose_video(in_video, "skeleton_only.mp4", mode="skeleton_only")
print(f"  skeleton-only:  wrote {n_skel} frames  →  skeleton_only.mp4")


In [ ]:
# --- Preview a handful of frames from each output video ---
# (Watching the full clip: open the Files tab in the left sidebar of
#  Colab, find with_skeleton.mp4 / skeleton_only.mp4, click ⋮ → Download.)

print("Overlay (original + skeleton) — preview frames:")
show_video_frames("with_skeleton.mp4", n_frames=6)

print("Skeleton only 🕺 — preview frames:")
show_video_frames("skeleton_only.mp4", n_frames=6)


In [ ]:
# --- Your dance signature: joint angles over time ---
if len(signature) == 0:
    print("No pose detected on any frame. Try a different video.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
    for col, color in zip(["l_knee", "r_knee"], ["#3b82f6", "#ef4444"]):
        axes[0].plot(signature["frame"], signature[col], lw=1.8, color=color, label=col)
    for col, color in zip(["l_elbow", "r_elbow"], ["#3b82f6", "#ef4444"]):
        axes[1].plot(signature["frame"], signature[col], lw=1.8, color=color, label=col)
    axes[0].set_ylabel("Knee angle (°)")
    axes[1].set_ylabel("Elbow angle (°)"); axes[1].set_xlabel("Frame")
    for ax in axes:
        ax.legend(loc="upper right"); ax.grid(alpha=0.3)
        ax.set_ylim(40, 200)
    plt.suptitle("🕺 Your dance signature — joint angles over time", fontsize=13)
    plt.tight_layout(); plt.show()

    # Stats: how much you moved
    rng = signature[["l_knee", "r_knee", "l_elbow", "r_elbow"]].max()           - signature[["l_knee", "r_knee", "l_elbow", "r_elbow"]].min()
    print("\nRange of motion (°):"); print(rng.round(1).to_string())


---

## ✏️ Exercises

**1 · Edge cases.** Run both YOLO11-pose and ViTPose on `cyclists.jpg` (multi-person scene). Which model handles multiple people better? Which one misses the occluded persons?

**2 · Pose comparison — yoga match.** Take two yoga photos (or two of yourself, same pose). Extract keypoints, normalise each by the bounding-box diagonal, then compute the **mean Euclidean distance** between the two keypoint sets. Lower = better match.

**3 · 🕺 Pose Karaoke — your turn.** Record a 10–20 second video of yourself doing **any** repeatable motion (squats, jumping jacks, the floss, even just waving). Run §5 and look at your dance signature — can you spot every rep in the joint-angle plot?

**4 · Multi-person karaoke.** Modify `process_pose_video` to draw **all** detected persons, not just the first. Re-run on a clip with two people. Bonus: assign each person a different skeleton colour.

Your code:


In [ ]:
# Your solution here
